<h1>Chapter 9 - Graph RAG: Recipe 3: Cypher Queries</h1>
<i>Building knowledge graphs and using them in RAG systems.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch09_graph_rag/9.3_cypher_queries.ipynb)

---

This notebook is for Chapter 9 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [12]:
%pip install python-dotenv neo4j

Note: you may need to restart the kernel to use updated packages.


## 1. Setup and Imports

Import required libraries and load environment variables.

In [13]:
from __future__ import annotations
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Load environment variables from .env file
load_dotenv()

True

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

In [14]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    if not uri or not user or not pwd:
        raise RuntimeError("Missing Neo4j env vars")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Test connection
driver = get_driver()
print("✓ Connected to Neo4j")

✓ Connected to Neo4j


## 3. List Clauses for a Specific SLA

Retrieve all clauses for one SLA, ordered by their original position in the document.

In [15]:
def list_clauses_for_sla(sla_id: str) -> List[Dict[str, Any]]:
    """Return all clauses for one SLA ordered by their original position."""
    cypher = """
    MATCH (s:SLA {id: $sla_id})-[:HAS_CLAUSE]->(cl:Clause)
    RETURN cl.order AS order,
           cl.title AS title,
           cl.text AS text
    ORDER BY order
    """
    driver = get_driver()
    with driver.session() as session:
        response = session.run(cypher, sla_id=sla_id)
        records = [r.data() for r in response]
        return records

# Execute
records_clauses = list_clauses_for_sla(sla_id="SLA1")
print(f"✓ Found {len(records_clauses)} clauses for SLA1")
for i, record in enumerate(records_clauses[:3], 1):
    print(f"  {i}. Order {record['order']}: {record['title']}")

✓ Found 13 clauses for SLA1
  1. Order 1: 1. Service Description
  2. Order 2: 2. Availability and Uptime Commitment
  3. Order 3: 3. Response Times and Incident Handling


## 5. Find Clauses of a Specific Type

List all clauses of a specific type (e.g., "Termination") across all suppliers and SLAs.

In [16]:
def clauses_of_type(clause_type: str) -> List[Dict[str, Any]]:
    """List all clauses of a specific ClauseType across suppliers."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: $clause_type})
    RETURN c.name AS company,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.title AS clause_title,
           cl.text AS clause_text
    ORDER BY company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, clause_type=clause_type)]

# Execute
records_clause_types = clauses_of_type(clause_type="Termination")
print(f"✓ Found {len(records_clause_types)} Termination clauses")
for i, record in enumerate(records_clause_types[:3], 1):
    print(f"  {i}. {record['company']} - {record['sla_id']}: {record['clause_title']}")

✓ Found 1 Termination clauses
  1. Alpha Cloud GmbH - SLA1: 10. Termination


## 6. High-Spend Companies Missing Termination Clauses

Identify companies with high spend whose SLAs lack termination clauses - a potential risk indicator.

In [17]:
def high_spend_missing_termination(min_spend: float) -> List[Dict[str, Any]]:
    """Return companies above min_spend whose SLAs lack a termination clause."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)
    WHERE c.spend_2024 > $min_spend
    OPTIONAL MATCH (s)-[:HAS_CLAUSE]->(cl:Clause)-[:OF_TYPE]->(t:ClauseType {name: "Termination"})
    WITH c, s, count(cl) AS num_termination
    WHERE num_termination = 0
    RETURN c.name AS company,
           c.spend_2024 AS spend_2024,
           s.id AS sla_id
    ORDER BY spend_2024 DESC
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, min_spend=min_spend)]

# Execute
records_min_spend = high_spend_missing_termination(min_spend=500000)
print(f"✓ Found {len(records_min_spend)} high-spend companies without termination clauses")
for i, record in enumerate(records_min_spend[:5], 1):
    spend = record['spend_2024']
    print(f"  {i}. {record['company']} - €{spend:,.2f}" if spend else f"  {i}. {record['company']}")

✓ Found 1 high-spend companies without termination clauses
  1. DataStore BV - €800,000.00


## 7. Search Availability Clauses

Inspect availability clauses across suppliers, searching for specific SLA commitments (e.g., "99.9%" uptime).

In [18]:
def availability_clauses(search_phrase: str) -> List[Dict[str, Any]]:
    """Inspect availability clauses per supplier."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: "Availability"})
    WHERE toLower(cl.text) CONTAINS toLower($phrase)
    RETURN c.name AS company,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.text AS availability_text
    ORDER BY company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, phrase=search_phrase)]

# Execute
availability_clauses_records = availability_clauses(search_phrase="99.9")
print(f"✓ Found {len(availability_clauses_records)} availability clauses containing '99.9'")
for i, record in enumerate(availability_clauses_records[:3], 1):
    print(f"  {i}. {record['company']} - {record['sla_id']}")
    text_preview = record['availability_text'][:100] + "..." if len(record['availability_text']) > 100 else record['availability_text']
    print(f"     {text_preview}")

✓ Found 1 availability clauses containing '99.9'
  1. Alpha Cloud GmbH - SLA1
     The supplier commits to a **monthly uptime percentage of 99.9 percent**. Downtime refers to interval...


## 8. EU Data Protection Clauses by Country

Retrieve data protection clauses for suppliers located in specific EU countries, useful for GDPR compliance reviews.

In [ ]:
def eu_data_protection_clauses(countries: list[str]):
    """Retrieve data protection clauses for suppliers in given EU countries."""
    cypher = """
    MATCH (c:Company)-[:LOCATED_AT]->(a:Address)
    WHERE a.country IN $countries
    MATCH (c)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: "DataProtection"})
    RETURN c.name AS company,
           a.country AS country,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.text AS data_protection_clause
    ORDER BY country, company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, countries=countries)]

countries = ["Germany", "France", "Netherlands"]
eu_data_protection_clauses_records = eu_data_protection_clauses(countries=countries)
print(f"✓ Found {len(eu_data_protection_clauses_records)} data protection clauses in {', '.join(countries)}")
for i, record in enumerate(eu_data_protection_clauses_records[:3], 1):
    print(f"  {i}. {record['company']} ({record['country']}) - {record['sla_id']}")

✓ Found 1 data protection clauses in Germany, France, Netherlands
  1. Alpha Cloud GmbH (Germany) - SLA1


## 9. Cleanup

Close the Neo4j driver connection.

In [20]:
driver.close()
print("✓ Connection closed")

✓ Connection closed
